In [1]:
import numpy as np
import pandas as pd

R_EARTH_KM = 6371.0

BASE_PATH = "data/compiled_model_ready/MR_cities_with_country_worldpop_v1.csv"
OUTPUT_PATH = "data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
cities = pd.read_csv(BASE_PATH)

print("Shape:", cities.shape)
print(cities.columns.tolist())
print(cities.head())

Shape: (509, 26)
['city', 'country', 'country_norm', 'lat', 'lon', 'crime_index', 'safety_index', 'crimeindex_2023', 'crimeindex_2020', 'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus', 'population', 'density_per_km2', 'num_cities_25km', 'sum_pop_25km', 'pop_gravity_25km', 'num_cities_50km', 'sum_pop_50km', 'pop_gravity_50km', 'num_cities_100km', 'sum_pop_100km', 'pop_gravity_100km', 'dist_to_nearest_large_city', 'pop_of_nearest_large_city']
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  crimeindex_2023  crimeindex_2020  \
0        

In [4]:
if "country_norm" not in cities.columns:
    cities["country_norm"] = cities["country"].astype(str).str.strip().str.lower()

In [5]:
numeric_cols = [
    "lat", "lon",
    "crime_index", "safety_index",
    "num_cities_25km", "sum_pop_25km", "pop_gravity_25km",
    "num_cities_50km", "sum_pop_50km", "pop_gravity_50km",
    "num_cities_100km", "sum_pop_100km", "pop_gravity_100km",
    "dist_to_nearest_large_city", "pop_of_nearest_large_city",
]

for col in numeric_cols:
    if col in cities.columns:
        cities[col] = pd.to_numeric(cities[col], errors="coerce")

cities = cities.dropna(subset=["lat", "lon", "crime_index", "safety_index"]).reset_index(drop=True)
print("After numeric clean:", cities.shape)

After numeric clean: (509, 26)


In [6]:
def haversine_vec(lat0, lon0, lats, lons):
    lat0 = np.radians(lat0)
    lon0 = np.radians(lon0)
    lats = np.radians(lats)
    lons = np.radians(lons)

    dlat = lats - lat0
    dlon = lons - lon0

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat0) * np.cos(lats) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R_EARTH_KM * c

In [7]:
K_LIST = [3, 5, 10]
DIST_BAND_KM = [50.0, 100.0, 250.0]
eps = 1e-3

In [8]:
feature_store = {
    "dist_nearest_labeled_city": [],
    "crime_nearest_labeled_city": [],
    "safety_nearest_labeled_city": [],
    "same_country_as_nearest_labeled": [],
    
    "avg_crime_k3": [],
    "avg_safety_k3": [],
    "avg_crime_k5": [],
    "avg_safety_k5": [],
    "avg_crime_k10": [],
    "avg_safety_k10": [],
    
    "wavg_crime_k5": [],
    "wavg_safety_k5": [],
    
    "num_labeled_within_50km": [],
    "num_labeled_within_100km": [],
    "num_labeled_within_250km": [],
    
    "avg_crime_same_country_k5": [],
    "avg_safety_same_country_k5": [],
    "num_same_country_within_250km": [],
}

In [10]:
city_lats = cities["lat"].values.astype(np.float32)
city_lons = cities["lon"].values.astype(np.float32)
city_crime = cities["crime_index"].values.astype(np.float32)
city_safety = cities["safety_index"].values.astype(np.float32)
city_country = cities["country_norm"].astype(str).values
city_names = cities["city"].astype(str).values

n = len(cities)
print("Labeled cities:", n)

Labeled cities: 509


In [11]:
for i in range(n):
    lat0 = city_lats[i]
    lon0 = city_lons[i]
    country0 = city_country[i]

    dists = haversine_vec(lat0, lon0, city_lats, city_lons)

    # exclude self
    dists[i] = np.inf

    nn_idx = np.argmin(dists)
    nn_dist = float(dists[nn_idx])

    feature_store["dist_nearest_labeled_city"].append(nn_dist)
    feature_store["crime_nearest_labeled_city"].append(float(city_crime[nn_idx]))
    feature_store["safety_nearest_labeled_city"].append(float(city_safety[nn_idx]))
    feature_store["same_country_as_nearest_labeled"].append(int(city_country[nn_idx] == country0))

    # sorted neighbors
    sorted_idx = np.argsort(dists)

    for k in K_LIST:
        k_idx = sorted_idx[:k]

        feature_store[f"avg_crime_k{k}"].append(float(np.mean(city_crime[k_idx])))
        feature_store[f"avg_safety_k{k}"].append(float(np.mean(city_safety[k_idx])))

    # weighted averages for k=5
    k = 5
    k_idx = sorted_idx[:k]
    k_dists = dists[k_idx]
    weights = 1.0 / (k_dists + eps)

    wavg_crime = np.sum(city_crime[k_idx] * weights) / np.sum(weights)
    wavg_safety = np.sum(city_safety[k_idx] * weights) / np.sum(weights)

    feature_store["wavg_crime_k5"].append(float(wavg_crime))
    feature_store["wavg_safety_k5"].append(float(wavg_safety))

    # counts in fixed distance bands
    for band in DIST_BAND_KM:
        count_band = int(np.sum(dists <= band))
        feature_store[f"num_labeled_within_{int(band)}km"].append(count_band)

    # same-country neighbors only
    same_country_mask = (city_country == country0)
    same_country_mask[i] = False

    same_country_idx = np.where(same_country_mask)[0]

    if len(same_country_idx) > 0:
        sc_dists = dists[same_country_idx]
        sc_sorted_local = same_country_idx[np.argsort(sc_dists)]

        k_sc = min(5, len(sc_sorted_local))
        sc_top = sc_sorted_local[:k_sc]

        feature_store["avg_crime_same_country_k5"].append(float(np.mean(city_crime[sc_top])))
        feature_store["avg_safety_same_country_k5"].append(float(np.mean(city_safety[sc_top])))

        sc_within_250 = int(np.sum(dists[same_country_idx] <= 250.0))
        feature_store["num_same_country_within_250km"].append(sc_within_250)
    else:
        feature_store["avg_crime_same_country_k5"].append(np.nan)
        feature_store["avg_safety_same_country_k5"].append(np.nan)
        feature_store["num_same_country_within_250km"].append(0)

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{n}")

Processed 50/509
Processed 100/509
Processed 150/509
Processed 200/509
Processed 250/509
Processed 300/509
Processed 350/509
Processed 400/509
Processed 450/509
Processed 500/509


In [12]:
for k, vals in feature_store.items():
    cities[k] = vals

print(cities.shape)
cities.head()

(509, 44)


,city,country,country_norm,lat,lon,crime_index,safety_index,crimeindex_2023,crimeindex_2020,safetyindex_2020,...,avg_crime_k10,avg_safety_k10,wavg_crime_k5,wavg_safety_k5,num_labeled_within_50km,num_labeled_within_100km,num_labeled_within_250km,avg_crime_same_country_k5,avg_safety_same_country_k5,num_same_country_within_250km
0,Caracas,Venezuela,venezuela,10.506093,-66.914601,83.6,16.4,83.76,84.49,15.51,...,63.900002,36.099998,67.918312,32.081684,0,0,0,NaN,NaN,0
1,Pretoria,South Africa,south africa,-25.745928,28.187910,82.0,18.0,76.86,77.49,22.51,...,69.574005,30.425999,75.717979,24.282015,0,1,1,76.703995,23.296000,1
2,Durban,South Africa,south africa,-29.861825,31.009909,81.0,19.0,76.86,77.49,22.51,...,69.673996,30.325998,74.345474,25.654528,0,0,0,76.903999,23.095999,0
3,Port Moresby,Papua New Guinea,papua new guinea,-9.474330,147.159950,80.7,19.3,80.79,81.93,18.07,...,54.200001,45.799999,60.770332,39.229660,0,0,0,NaN,NaN,0
4,Johannesburg,South Africa,south africa,-26.205000,28.049722,80.7,19.3,76.86,77.49,22.51,...,69.704002,30.296000,77.340050,22.659943,0,1,1,76.964005,23.035999,1


In [13]:
log_cols = [
    "sum_pop_25km", "pop_gravity_25km",
    "sum_pop_50km", "pop_gravity_50km",
    "sum_pop_100km", "pop_gravity_100km",
    "pop_of_nearest_large_city",
    "num_cities_25km", "num_cities_50km", "num_cities_100km",
    "num_labeled_within_50km", "num_labeled_within_100km", "num_labeled_within_250km",
    "num_same_country_within_250km",
]

for col in log_cols:
    if col in cities.columns:
        cities[f"log1p_{col}"] = np.log1p(cities[col].fillna(0))

# distance features: leave raw, but can also log them
dist_cols = [
    "dist_to_nearest_large_city",
    "dist_nearest_labeled_city",
]

for col in dist_cols:
    if col in cities.columns:
        cities[f"log1p_{col}"] = np.log1p(cities[col].fillna(0))

In [14]:
country_rel_cols = [
    "sum_pop_50km",
    "sum_pop_100km",
    "pop_gravity_50km",
    "pop_gravity_100km",
    "dist_to_nearest_large_city",
]

for col in country_rel_cols:
    if col in cities.columns:
        grp = cities.groupby("country_norm")[col]
        mean_col = grp.transform("mean")
        std_col = grp.transform("std").replace(0, np.nan)

        cities[f"{col}_country_z"] = (cities[col] - mean_col) / std_col
        cities[f"{col}_country_z"] = cities[f"{col}_country_z"].fillna(0.0)

In [15]:
new_feature_cols = [
    "dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "same_country_as_nearest_labeled",
    "avg_crime_k3", "avg_safety_k3",
    "avg_crime_k5", "avg_safety_k5",
    "avg_crime_k10", "avg_safety_k10",
    "wavg_crime_k5", "wavg_safety_k5",
    "num_labeled_within_50km", "num_labeled_within_100km", "num_labeled_within_250km",
    "avg_crime_same_country_k5", "avg_safety_same_country_k5",
    "num_same_country_within_250km",
]

print(cities[new_feature_cols].describe().T)

                                 count        mean         std        min  \
dist_nearest_labeled_city        509.0  182.757840  256.126075   0.000000   
crime_nearest_labeled_city       509.0   46.362299   15.450567  15.100000   
safety_nearest_labeled_city      509.0   53.637701   15.450567  16.400000   
same_country_as_nearest_labeled  509.0    0.758350    0.428505   0.000000   
avg_crime_k3                     509.0   45.465455   12.913631  14.200000   
avg_safety_k3                    509.0   54.534545   12.913631  18.766666   
avg_crime_k5                     509.0   45.566366   11.614622  14.700000   
avg_safety_k5                    509.0   54.433634   11.614622  22.276001   
avg_crime_k10                    509.0   45.523593   10.049688  23.076000   
avg_safety_k10                   509.0   54.476407   10.049689  27.792999   
wavg_crime_k5                    509.0   46.047704   12.932095  14.737869   
wavg_safety_k5                   509.0   53.952296   12.932096  22.261204   

In [16]:
corr_cols = [c for c in new_feature_cols if c in cities.columns]
corr_df = cities[corr_cols + ["safety_index"]].corr(numeric_only=True)[["safety_index"]].sort_values(
    by="safety_index", ascending=False
)
print(corr_df)

                                 safety_index
safety_index                         1.000000
wavg_safety_k5                       0.705954
avg_safety_same_country_k5           0.670982
avg_safety_k10                       0.640839
avg_safety_k5                        0.634749
safety_nearest_labeled_city          0.631711
avg_safety_k3                        0.619141
num_labeled_within_250km             0.144051
num_labeled_within_100km             0.042319
num_same_country_within_250km        0.042031
same_country_as_nearest_labeled     -0.039943
num_labeled_within_50km             -0.064814
dist_nearest_labeled_city           -0.086883
avg_crime_k3                        -0.619141
crime_nearest_labeled_city          -0.631711
avg_crime_k5                        -0.634749
avg_crime_k10                       -0.640839
avg_crime_same_country_k5           -0.670982
wavg_crime_k5                       -0.705954


In [17]:
knn_cols = [
    "dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "same_country_as_nearest_labeled",
    "avg_crime_k3", "avg_safety_k3",
    "avg_crime_k5", "avg_safety_k5",
    "avg_crime_k10", "avg_safety_k10",
    "wavg_crime_k5", "wavg_safety_k5",
    "num_labeled_within_50km", "num_labeled_within_100km", "num_labeled_within_250km",
    "avg_crime_same_country_k5", "avg_safety_same_country_k5",
    "num_same_country_within_250km",
]

print(cities[knn_cols].isna().sum())
print(cities[knn_cols].describe(percentiles=[0.25, 0.5, 0.75]).T)

dist_nearest_labeled_city           0
crime_nearest_labeled_city          0
safety_nearest_labeled_city         0
same_country_as_nearest_labeled     0
avg_crime_k3                        0
avg_safety_k3                       0
avg_crime_k5                        0
avg_safety_k5                       0
avg_crime_k10                       0
avg_safety_k10                      0
wavg_crime_k5                       0
wavg_safety_k5                      0
num_labeled_within_50km             0
num_labeled_within_100km            0
num_labeled_within_250km            0
avg_crime_same_country_k5          64
avg_safety_same_country_k5         64
num_same_country_within_250km       0
dtype: int64
                                 count        mean         std        min  \
dist_nearest_labeled_city        509.0  182.757840  256.126075   0.000000   
crime_nearest_labeled_city       509.0   46.362299   15.450567  15.100000   
safety_nearest_labeled_city      509.0   53.637701   15.450567  16.40000

In [22]:
# Same country averages: fall back to global k=5 averages where missing
for col_sc, col_global in [
    ("avg_crime_same_country_k5", "avg_crime_k5"),
    ("avg_safety_same_country_k5", "avg_safety_k5"),
]:
    if col_sc in cities.columns and col_global in cities.columns:
        cities[col_sc] = cities[col_sc].fillna(cities[col_global])

In [23]:
if "num_same_country_within_250km" in cities.columns:
    cities["num_same_country_within_250km"] = (
        cities["num_same_country_within_250km"].fillna(0).astype(int)
    )

In [24]:
log_cols = [
    # labeled-city counts
    "num_labeled_within_50km",
    "num_labeled_within_100km",
    "num_labeled_within_250km",
    "num_same_country_within_250km",
    # worldcities counts & populations
    "num_cities_25km",
    "num_cities_50km",
    "num_cities_100km",
    "sum_pop_25km",
    "sum_pop_50km",
    "sum_pop_100km",
    "pop_gravity_25km",
    "pop_gravity_50km",
    "pop_gravity_100km",
    "pop_of_nearest_large_city",
]

for col in log_cols:
    if col in cities.columns:
        cities[f"log1p_{col}"] = np.log1p(cities[col].fillna(0))

In [25]:
for col in ["dist_nearest_labeled_city", "dist_to_nearest_large_city"]:
    if col in cities.columns:
        cities[f"log1p_{col}"] = np.log1p(cities[col].fillna(0))

In [26]:
print(cities[[
    "avg_crime_same_country_k5",
    "avg_safety_same_country_k5",
    "num_same_country_within_250km"
]].isna().sum())

print(cities.filter(regex="^log1p_").describe().T.head(15))

avg_crime_same_country_k5        0
avg_safety_same_country_k5       0
num_same_country_within_250km    0
dtype: int64
                                     count       mean       std        min  \
log1p_sum_pop_25km                   509.0  14.153274  2.165049   0.000000   
log1p_pop_gravity_25km               509.0  14.450230  4.238189   0.000000   
log1p_sum_pop_50km                   509.0  14.590094  1.662561   0.000000   
log1p_pop_gravity_50km               509.0  14.511396  4.085797   0.000000   
log1p_sum_pop_100km                  509.0  15.109305  1.505004   0.000000   
log1p_pop_gravity_100km              509.0  14.531891  4.041110   0.000000   
log1p_pop_of_nearest_large_city      509.0  14.402283  0.984156  13.124567   
log1p_num_cities_25km                509.0   2.358768  0.992712   0.000000   
log1p_num_cities_50km                509.0   3.040858  1.094058   0.000000   
log1p_num_cities_100km               509.0   3.763010  1.164619   0.000000   
log1p_num_labeled_within

In [28]:
print(cities.isna())

      city  country  country_norm    lat    lon  crime_index  safety_index  \
0    False    False         False  False  False        False         False   
1    False    False         False  False  False        False         False   
2    False    False         False  False  False        False         False   
3    False    False         False  False  False        False         False   
4    False    False         False  False  False        False         False   
..     ...      ...           ...    ...    ...          ...           ...   
504  False    False         False  False  False        False         False   
505  False    False         False  False  False        False         False   
506  False    False         False  False  False        False         False   
507  False    False         False  False  False        False         False   
508  False    False         False  False  False        False         False   

     crimeindex_2023  crimeindex_2020  safetyindex_2020  ...  \

In [29]:
OUTPUT_PATH = "data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"
cities.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)

Saved: data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv


In [1]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path
 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_Features_v2.ipynb to pdf
[NbConvertApp] Writing 83907 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 59872 bytes to Geo_Features_v2.pdf
